# LLM Eval - End-to-End Test

This notebook tests the complete llm_eval system:
1. **PromptRegistry** - Register and version prompts
2. **LLMTracer** - Trace LLM calls
3. **TraceStore** - Query traces and sessions
4. **Annotations** - Rate and annotate outputs
5. **Dashboard** - Launch Streamlit UI

In [ ]:
import sys
sys.path.insert(0, '..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv("../.env")

In [ ]:
# Import everything from llm_eval
from llm_eval import (
    Prompt,
    Trace,
    Annotation,
    TraceStore,
    PromptRegistry,
    LLMTracer,
)
from llm_eval.utils.models import ModelSelector

print("All imports successful!")

## 1. Setup Database

All data is stored in a local SQLite database file.

In [ ]:
DB_PATH = "e2e_test.db"

# Initialize components
store = TraceStore(DB_PATH)
registry = PromptRegistry(DB_PATH)

print(f"Database: {DB_PATH}")
print("Store and Registry initialized!")

## 2. Register Prompts

Register versioned prompts with automatic deduplication.

In [ ]:
# Register a summarization prompt (v1)
summarizer_v1 = registry.register(
    name="summarizer",
    template="""You are a helpful assistant. Summarize the following text concisely.

Text: {text}

Summary:""",
    description="Initial summarizer prompt"
)
print(f"Registered: {summarizer_v1.name} v{summarizer_v1.version}")

In [ ]:
# Register an improved version (v2)
summarizer_v2 = registry.register(
    name="summarizer",
    template="""You are an expert summarizer. Create a concise, accurate summary that captures the key points.

Text: {text}

Summary (2-3 sentences max):""",
    description="Improved prompt with expert framing and length constraint"
)
print(f"Registered: {summarizer_v2.name} v{summarizer_v2.version}")

In [ ]:
# Register a Q&A prompt
qa_prompt = registry.register(
    name="qa_assistant",
    template="""You are a helpful assistant. Answer the question accurately and concisely.

Question: {question}

Answer:""",
    description="Simple Q&A prompt"
)
print(f"Registered: {qa_prompt.name} v{qa_prompt.version}")

In [ ]:
# List all versions of summarizer
versions = registry.list_versions("summarizer")
print(f"\nSummarizer has {len(versions)} versions:")
for v in versions:
    print(f"  v{v.version}: {v.description}")

## 3. Create Tracer and Make LLM Calls

The tracer captures all LLM interactions automatically.

In [ ]:
# Create tracer for a new project
tracer = LLMTracer(
    db_path=DB_PATH,
    project="e2e_test",
    session_id="session_001"
)

print(f"Tracer created!")
print(f"  Project: {tracer.project}")
print(f"  Session: {tracer.session_id}")

In [ ]:
# Set prompt context (REQUIRED before making LLM calls)
tracer.set_prompt_context("summarizer", version=1)
print(f"Prompt context: {tracer._prompt_name} v{tracer._prompt_version}")

In [ ]:
# Get model with tracer attached
llm = ModelSelector.get_model("gpt-4o-mini", callbacks=[tracer])
print("Model ready with tracer attached!")

In [ ]:
# Make first LLM call
messages1 = [
    {"role": "system", "content": "You are a helpful assistant. Summarize concisely."},
    {"role": "user", "content": "Summarize: Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve over time without being explicitly programmed. It uses algorithms to identify patterns in data and make predictions or decisions."}
]

response1 = llm.invoke(messages1)
print(f"Response 1: {response1.content}")

In [ ]:
# Make second call with v2 prompt
tracer.set_prompt_context("summarizer", version=2)

messages2 = [
    {"role": "system", "content": "You are an expert summarizer. Be concise."},
    {"role": "user", "content": "Summarize: The Internet of Things (IoT) refers to the network of physical devices embedded with sensors, software, and connectivity that enables them to collect and exchange data. IoT has applications in smart homes, healthcare, manufacturing, and transportation."}
]

response2 = llm.invoke(messages2)
print(f"Response 2: {response2.content}")

In [ ]:
# Start new session for Q&A
tracer.new_session("qa_session_001")
tracer.set_prompt_context("qa_assistant")

messages3 = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": "What is the capital of Japan?"}
]

response3 = llm.invoke(messages3)
print(f"Response 3: {response3.content}")

In [ ]:
# One more Q&A call
messages4 = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": "What is 15 * 23?"}
]

response4 = llm.invoke(messages4)
print(f"Response 4: {response4.content}")

## 4. Query Traces and Sessions

View all captured traces and session summaries.

In [ ]:
# Get all sessions
sessions = store.get_sessions()
print(f"Total sessions: {len(sessions)}\n")

print(f"{'Session':<25} {'Project':<15} {'Traces':>7} {'Tokens':>10} {'Success':>8} {'Errors':>7}")
print("-" * 80)
for s in sessions:
    name = s['session_id'][:23] + ".." if len(s['session_id']) > 25 else s['session_id']
    print(f"{name:<25} {s['project']:<15} {s['trace_count']:>7} {s['total_tokens'] or 0:>10} {s['success_count']:>8} {s['error_count']:>7}")

In [ ]:
# Get traces from first session
traces = store.get_traces({"session_id": "session_001"})
print(f"Traces in session_001: {len(traces)}\n")

for i, t in enumerate(traces):
    print(f"Trace {i+1}:")
    print(f"  ID: {t['id'][:12]}...")
    print(f"  Prompt: {t['prompt_name']} v{t['prompt_version']}")
    print(f"  Model: {t['model_name']}")
    print(f"  Tokens: {t['input_tokens']} in / {t['output_tokens']} out = {t['total_tokens']} total")
    print(f"  Latency: {t['latency_ms']}ms")
    print(f"  Status: {t['status']}")
    print(f"  Output: {t['output_content'][:100]}..." if len(t['output_content']) > 100 else f"  Output: {t['output_content']}")
    print()

In [ ]:
# Get a specific trace with full details
trace_id = traces[0]["id"]
full_trace = store.get_trace(trace_id)

print("Full trace details:")
print(f"  Input messages: {full_trace['input_messages']}")
print(f"  Metadata: {full_trace['metadata']}")

## 5. Add Annotations

Rate outputs as good/bad and add notes.

In [ ]:
# Annotate the first trace as good
trace_id_1 = traces[0]["id"]

annotation1 = Annotation(
    trace_id=trace_id_1,
    rating="good",
    notes="Accurate and concise summary",
    failure_category="",
    annotator="test_user"
)

store.save_annotation(annotation1.model_dump())
print(f"Annotated trace {trace_id_1[:8]}... as GOOD")

In [ ]:
# Annotate the second trace as bad (for demo purposes)
if len(traces) > 1:
    trace_id_2 = traces[1]["id"]
    
    annotation2 = Annotation(
        trace_id=trace_id_2,
        rating="bad",
        notes="Summary missed key points about healthcare application",
        failure_category="incomplete_coverage",
        annotator="test_user"
    )
    
    store.save_annotation(annotation2.model_dump())
    print(f"Annotated trace {trace_id_2[:8]}... as BAD")

In [ ]:
# Retrieve annotations
for t in traces:
    ann = store.get_annotation(t["id"])
    if ann:
        print(f"Trace {t['id'][:8]}...: {ann['rating'].upper()} - {ann['notes']}")
    else:
        print(f"Trace {t['id'][:8]}...: Not annotated")

## 6. Filter and Query

Advanced filtering options.

In [ ]:
# Filter by project
project_traces = store.get_traces({"project": "e2e_test"})
print(f"Traces in 'e2e_test' project: {len(project_traces)}")

In [ ]:
# Filter by prompt name
summarizer_traces = store.get_traces({"prompt_name": "summarizer"})
print(f"Traces using 'summarizer' prompt: {len(summarizer_traces)}")

In [ ]:
# Get sessions for specific project
project_sessions = store.get_sessions(project="e2e_test")
print(f"Sessions in 'e2e_test': {len(project_sessions)}")
for s in project_sessions:
    print(f"  - {s['session_id']}: {s['trace_count']} traces")

## 7. Summary Statistics

In [ ]:
# Calculate summary stats
all_traces = store.get_traces({})
all_sessions = store.get_sessions()

total_tokens = sum(t['total_tokens'] for t in all_traces)
total_latency = sum(t['latency_ms'] for t in all_traces)
avg_latency = total_latency / len(all_traces) if all_traces else 0

# Count annotations
good_count = 0
bad_count = 0
unannotated = 0
for t in all_traces:
    ann = store.get_annotation(t['id'])
    if ann:
        if ann['rating'] == 'good':
            good_count += 1
        else:
            bad_count += 1
    else:
        unannotated += 1

print("=" * 50)
print("SUMMARY STATISTICS")
print("=" * 50)
print(f"Total Sessions:     {len(all_sessions)}")
print(f"Total Traces:       {len(all_traces)}")
print(f"Total Tokens:       {total_tokens:,}")
print(f"Avg Latency:        {avg_latency:.0f}ms")
print(f"")
print(f"Annotations:")
print(f"  Good:             {good_count}")
print(f"  Bad:              {bad_count}")
print(f"  Unannotated:      {unannotated}")
if good_count + bad_count > 0:
    print(f"  Pass Rate:        {good_count / (good_count + bad_count) * 100:.1f}%")
print("=" * 50)

## 8. Launch Dashboard

Run this in your terminal to launch the Streamlit dashboard:

```bash
cd /path/to/phase_3_evals
streamlit run llm_eval/dashboard.py -- --db-path notebooks/e2e_test.db
```

Or run from this notebook:

In [ ]:
import os
print("To launch the dashboard, run this command in your terminal:\n")
print(f"cd {os.path.dirname(os.getcwd())}")
print(f"streamlit run llm_eval/dashboard.py -- --db-path notebooks/e2e_test.db")

## Cleanup (Optional)

In [ ]:
# Uncomment to delete test database
# import os
# os.remove("e2e_test.db")
# print("Deleted e2e_test.db")